<a href="https://colab.research.google.com/github/nikitask14/adult-income-classification/blob/main/adult_income_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


*  Importing Pandas, numpy library
*  Importing the skicit library - sklearn and importing the dataset loader - fetch_openml




In [77]:
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np

### Dataset loading note

The original plan was to load the Adult dataset using `fetch_openml`, but OpenML repeatedly returned a 504 Gateway Timeout.

To avoid delaying the project, I loaded the same Adult/Census Income dataset from the UCI Machine Learning Repository using `ucimlrepo`.



In [78]:
!pip install ucimlrepo


In [79]:
from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=20)

X = adult.data.features
y = adult.data.targets

After loading:
- `X` is a Pandas DataFrame with shape `(48842, 14)`
- `y` should be a 1D Pandas Series with shape `(48842,) but the UCI loader initially returned the target as a one-column DataFrame of shape `(48842, 1), so the target converted to a Series before continuing.

In [80]:
type(X)
X.shape


(48842, 14)

In [81]:
type(y)
y.shape


(48842, 1)

In [82]:
y.columns

Index(['income'], dtype='object')

In [83]:
y = y["income"]



In [84]:
type(y)
y.shape

(48842,)

Using info() method to get a comprehensive summary about the dataset.


1.  It tells us about the number of rows and columns.
2.  How many missing values are present
3.  What the numerical and categorical features are.
4.  The Non-Null tells us the number of actual values present for each of the feature columns.
5.  Columns with 48,842 non-null (like age or education) are completely full.



In [85]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48568 non-null  object
dtypes: int64(6), object(8)
memory usage: 5.2+ MB


Now, checking the number of null values in for each of the feature columns and sorting them by features with highest missing values to lowest.

In [86]:
X.isnull().sum().sort_values(ascending = False)

,0
occupation,966
workclass,963
native-country,274
education,0
fnlwgt,0
age,0
marital-status,0
education-num,0
race,0
relationship,0


occupation has 2809 missing values, workclass	has 2799 missing values, native-country	has 857 missing values.





In [87]:
X.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba


 DATA AUDIT  
*   Check for class imbalance
*   fnlwgt(doesn't seem to carry a lot of meaning), marital status not of much consequnce if earning or not, Race is controversial,
*   Check for data leakage - found none upfront.
*  Education and education-num seem similar - maybe keep education-num and drop out education to save dimensionality.








In [88]:
X["native-country"].unique()

array(['United-States', 'Cuba', 'Jamaica', 'India', '?', 'Mexico',
       'South', 'Puerto-Rico', 'Honduras', 'England', 'Canada', 'Germany',
       'Iran', 'Philippines', 'Italy', 'Poland', 'Columbia', 'Cambodia',
       'Thailand', 'Ecuador', 'Laos', 'Taiwan', 'Haiti', 'Portugal',
       'Dominican-Republic', 'El-Salvador', 'France', 'Guatemala',
       'China', 'Japan', 'Yugoslavia', 'Peru',
       'Outlying-US(Guam-USVI-etc)', 'Scotland', 'Trinadad&Tobago',
       'Greece', 'Nicaragua', 'Vietnam', 'Hong', 'Ireland', 'Hungary',
       'Holand-Netherlands', nan], dtype=object)

In [89]:
X["occupation"].unique()

array(['Adm-clerical', 'Exec-managerial', 'Handlers-cleaners',
       'Prof-specialty', 'Other-service', 'Sales', 'Craft-repair',
       'Transport-moving', 'Farming-fishing', 'Machine-op-inspct',
       'Tech-support', '?', 'Protective-serv', 'Armed-Forces',
       'Priv-house-serv', nan], dtype=object)

The UCI loader preserved some values differently from OpenML. Missing categorical values appeared party as "?", and party as NaN. I corrected this so that all missing values show as NaN.

In [90]:
X = X.replace("?",np.nan)
X["occupation"].unique()

array(['Adm-clerical', 'Exec-managerial', 'Handlers-cleaners',
       'Prof-specialty', 'Other-service', 'Sales', 'Craft-repair',
       'Transport-moving', 'Farming-fishing', 'Machine-op-inspct',
       'Tech-support', nan, 'Protective-serv', 'Armed-Forces',
       'Priv-house-serv'], dtype=object)

In [91]:
y.value_counts()

,count
income,
<=50K,24720
<=50K.,12435
>50K,7841
>50K.,3846


The target appeared as four strings variants because some labels had trailing periods, and to correct this, we need to strip the trailing periods from the category names so the duplicate rows merge together.

In [92]:
y = y.str.rstrip('.')
y.value_counts()

,count
income,
<=50K,37155
>50K,11687


High class imbalance

---


Class 0 = 76%
Class 1 = 24%

We need to stratify the train_test split.

In [93]:
retained_columns = ["occupation", "workclass", "native-country", "age",	"marital-status", "education-num", "race",
                    "relationship", "sex", "capital-gain", "capital-loss", "hours-per-week"]

In [94]:
type(retained_columns)

list

In [95]:
X_selected = X[retained_columns]
X_selected.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   occupation      46033 non-null  object
 1   workclass       46043 non-null  object
 2   native-country  47985 non-null  object
 3   age             48842 non-null  int64 
 4   marital-status  48842 non-null  object
 5   education-num   48842 non-null  int64 
 6   race            48842 non-null  object
 7   relationship    48842 non-null  object
 8   sex             48842 non-null  object
 9   capital-gain    48842 non-null  int64 
 10  capital-loss    48842 non-null  int64 
 11  hours-per-week  48842 non-null  int64 
dtypes: int64(5), object(7)
memory usage: 4.5+ MB


We now have 48842 entries.

> 12 feature columns, positive class is income >$50. High class imbalance, no leakage, raw/high cardinality features removed. Redundant features removed.



In [96]:
from sklearn.model_selection import train_test_split

Next, we are creating a training test split and then diving the training split further to create a validation set to decide on the how well the model performs on unseen data before finally checking on how the model performs on test(unseen) data.

It also ensures there is no data leakage.

In [97]:
X_train_val, X_test_final, y_train_val, y_test_final = train_test_split(
    X_selected,
    y,
    test_size = 0.20,
    random_state = 42,
    stratify = y
)

In [98]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size = 0.25,
    random_state=42,
    stratify = y_train_val

)

Now, that we have split the data, we will now identify the numerical and categorical columns before preprocessing as they require different preprocessing techniques.

In [99]:
numerical_columns = X_selected.select_dtypes(include = "number").columns
numerical_columns
# len(numerical_columns)

Index(['age', 'education-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='object')

In [100]:
categorical_columns = X_selected.select_dtypes(include = "object").columns
categorical_columns
# len(categorical_columns)


Index(['occupation', 'workclass', 'native-country', 'marital-status', 'race',
       'relationship', 'sex'],
      dtype='object')

In [101]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder)
from sklearn.compose import ColumnTransformer


In [102]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy = "median")),
    ("scale", StandardScaler())
])

In [103]:
cat_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy = "most_frequent")),
    "encoder", OneHotEncoder(handle_unknown="ignore")
])

In [104]:
preprocessor = ColumnTransformer(
    [
        ("numerical",num_pipeline, numerical_columns),
        ("cat", cat_pipeline, categorical_columns)
    ]
)